In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tifffile
import ipywidgets as W
from ipywidgets import interact, widgets, fixed

# Add src to path if running from notebooks folder
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
    
from scene import generate_single_cell_3d
from render import N_POOLS, render_image, to_rgb
from tape import Tape
from plot import plot_surface_xyz_inline, plot_surface_xyz_html, plot_surface_rgb_html, ortho, ortho_rgb, tau_cmap
from parameter import P

%matplotlib inline

# 1) Synthetic cell generation

In [ ]:
SEED = None
N_CAND = 1500
SIZE = 201
TILE = 256
K = 20

# lowest SH degree. 0 = pure size (absorbed by the volume normalisation),
# 1 = shifts the centroid off the seed. 2 = lowest true shape mode.
L_MIN = 2
# 4 is the ceiling of the hardcoded Cartesian forms
L = 4

# microns per LATERAL voxel
UM_PER_VOX = 0.325
# z voxels this many times COARSER than lateral
Z_RATIO = 1.0
# (sz, sy, sx) = voxel SIZE per axis, in lateral units
SPACING = (Z_RATIO, 1.0, 1.0)   
VOL = (128, 128, 128)
IMG = (128, 128, 3)


# 1) CELL GEOMETRY

RADIUS = P(4.0, 48.0, 0.5, 24.0, "radius", note="radius", 
           comment = "equivalent-sphere radius; volume is normalised to (4/3)pi R^3 whatever the roughness."
           )

ROUGH  = P(0.0, 0.50, 0.01, 0.25, "rough", note="rough",
           comment = "SD of log-radius: 0.25 ~ +-25% radial wobble. Orthogonal to radius and beta."
           )

BETA   = P(0.5, 4.0, 0.1, 1.9, "beta", note="beta",
           comment = "spectral tilt at FIXED total amplitude. Large -> a few fat lobes;\
small -> finer crenulation. Does NOT change how rough the cell is."
           )

ELONG  = P(0.3, 3.0, 0.05, 1.6, "elong", note="elong",
           comment = "volume-preserving aspect ratio. >1 prolate (rod), 1 = sphere, <1 oblate."
           )

# 2) NUCLEUS

NUC_FRAC   = P(0.10, 0.80, 0.01, 0.25, "nuc_frac",   note="nuc frac",
           comment = "nucleus:cell equivalent-RADIUS ratio -> volume ratio is nuc_frac**3."
           )

NUC_ROUGH  = P(0.0, 0.50, 0.01, 0.50, "nuc_rough",  note="nuc rough",
           comment = "nuclear log-radius SD (0.25 ~ +-25% radial wobble.) Orthogonal to radius and beta"
           )

# nuclear spectral tilt. Currently HARDCODED as beta+1.5 -- see caveat below.
NUC_BETA   = P(0.5, 6.0, 0.1, 3.2, "nuc_beta",    note="nuc beta",
           comment = "nuclear spectral tilt. Large -> a few fat lobes;\
small -> finer crenulation"
           )

NUC_CORR   = P(0.0, 1.0, 0.05, 0.40, "nuc_corr",   note="nuc corr",
           comment = "how much the nuclear outline mirrors the cell's"
           )

NUC_OFFSET = P(0.0, 1.0, 0.05, 1., "nuc_offset", note="nuc offset",
           comment = "nuclear displacement as a fraction of the free cytoplasmic room. \
0 puts the nucleus exactly on the tessellation seed -> trivially recoverable."
           )

RIM        = P(0.0, 8.0, 0.5, 0.0, "rim",          note="rim",
           comment = "minimum cytoplasm between nuclear and plasma membrane, in lateral voxels."
           )

# 3) ORIENTATION   (ZYZ: polar+azim aim the long axis, roll spins about it)

POLAR_DEG = P(0.0, 180.0, 5.0, 65.0, "polar_deg", note="polar")
AZIM_DEG = P(0.0, 360.0, 5.0, 25.0, "azim_deg",  note="azim")
ROLL_DEG = P(0.0, 360.0, 5.0,  0.0, "roll_deg",  note="roll")


# 4) DRAW THE TAPE

tape = Tape(seed=SEED, size=SIZE, K=K)
tape.draw(tile=TILE, n_cand=N_CAND, Pool=N_POOLS)
tape.draw3d(vol=VOL, n_cand=N_CAND, Pool=N_POOLS, l_min=L_MIN, L=L)
tape.drawSensor(shape=IMG)

## 1.1) Cell shape gen

In [ ]:
cell = generate_single_cell_3d(tape, i=0, size=VOL, spacing=SPACING, radius=RADIUS.v, nuc_frac=NUC_FRAC.v, rough=ROUGH.v, elong=ELONG.v, beta=BETA.v,
          polar_deg=POLAR_DEG.v, azim_deg=AZIM_DEG.v, roll_deg=ROLL_DEG.v, rim=RIM.v, nuc_corr=NUC_CORR.v, nuc_offset=NUC_OFFSET.v, beta_nuc = NUC_BETA.v,
          rough_nuc = NUC_ROUGH.v)

In [ ]:
fig = plot_surface_xyz_inline(cell = cell, elong = ELONG.v, 
                        polar_deg = POLAR_DEG.v, azim_deg = AZIM_DEG.v, roll_deg = ROLL_DEG.v)
fig = plot_surface_xyz_html(cell = cell, elong = ELONG.v, 
                        polar_deg = POLAR_DEG.v, azim_deg = AZIM_DEG.v, roll_deg = ROLL_DEG.v,
                        out_path = Path("../renders").resolve())

In [ ]:
lab = cell["cell"].astype(np.int8) + cell["nuc"]        # 0 background, 1 cytoplasm, 2 nucleus
ortho(lab, cmap="viridis", title="mask:  0 background   1 cytoplasm   2 nucleus")
plt.show()

## 1.2) Cell Marker expression

### 1.2.2) Marker espression

In [ ]:
cm, vmin, vmax = tau_cmap(float(cell["tau"][cell["cell"]].max()))
ortho(cell["tau"], cmap=cm, vmin=vmin, vmax=vmax, mask=cell["cell"],
      title=r"$\tau$   (-1 nucleus centre,  0 nuclear envelope,  +1 plasma membrane)")
plt.show()

In [ ]:
marker_config = dict(
    general = dict(
        scope = "general",
        amp=P(.1, 3, .1, 2, "amp"),
        polarity=P(-2, 2, .1, 0.2, "polarity"),
        pol_dir=(
            P(-np.pi, np.pi, .1, 0.0, "pol dir"),
            P(-np.pi, np.pi, .1, 1.0, "pol dir"),
            P(-np.pi, np.pi, .1, 2.0, "pol dir")
        ),
        fluorophores = {
            "r": "APC",
            "g": "FITC",
            "b": "PE",
        },
        detector = dict(
            AF_SCALE_UM = P(1., 100., 1., 25.0, "af scale",   note="af scale",
                            comment="spatial scale of the autofluorescence field, in microns"),
            AF_CV       = P(0., 1.5, .05, 0.45, "af cv",      note="af cv",
                            comment="relative SD of the autofluorescence field. 0 = perfectly flat background"),
            ILLUM_CV    = P(0., .5, .01, 0.06, "illum cv",    note="illum cv",
                            comment="random flat-field non-uniformity, as a fraction"),
            VIGNETTE    = P(0., .6, .05, 0.18, "vignette",    note="vignette",
                            comment="radial illumination falloff at the frame corners, as a fraction"),
            E_PER_UNIT  = P(1., 2000., 10., 120.0, "e per unit", note="e/unit",
                            comment="photoelectrons per unit of marker amplitude. THIS sets the shot-noise \
            level: doubling it halves the relative noise. amp x e_per_unit = photons per cell."),
            READ_E      = P(0., 50., .5, 2.5, "read noise",   note="read e-",
                            comment="camera read noise, electrons RMS. Dominates where the signal is dark."),
            DARK_E      = P(0., 200., 1., 5.0, "dark",        note="dark e-",
                            comment="dark current + stray light, in electrons"),
            ADU_PER_E   = P(.05, 10., .05, 0.5, "adu per e",  note="adu/e-",
                            comment="digitiser conversion gain"),
            OFFSET_ADU  = P(0., 2000., 10., 100.0, "offset",  note="offset",
                            comment="camera black level"),
            BIT_DEPTH   = 16
        )
    ),
    r = [
        dict(
            scope = "cluster",
            w=P(0, 1, .05, .80, "weight"),      s=P(0, 2, .05, 1.40, "strength"),
            mu=P(-1, 1.5, .05, 0.55, "mu"),     width=P(.05, 1.5, .05, 0.45, "width"),
            sharp=P(.5, 10, .5, 5.0, "sharp"),
            scale=P(.1, 1.5, .05, 0.35, "speckle um"),
            clust=P(.5, 8, .1, 2.00, "cluster um"),
            fill=P(.02, 1, .02, 0.22, "fill frac"),
            soft=P(.05, 1, .05, 0.25, "gate soft"),
        ),
        dict(
            scope = 'fibre',
            w=P(0, 1, .05, .30, "weight"),      s=P(0, 2, .05, 1.30, "strength"),
            mu=P(-1, 1.5, .05, 0.20, "mu"),     width=P(.05, 1.5, .05, 0.70, "width"),
            sharp=P(.5, 10, .5, 4.0, "sharp"),
            lam=P(.1, 1.5, .05, 0.25, "thickness um"),
            len=P(1, 20, .5, 6.0, "length um"),
        ),
        dict(
            scope = 'network',
            w=P(0, 1, .05, .40, "weight"),      s=P(0, 2, .05, 1.30, "strength"),
            mu=P(-1, 1.5, .05, 0.50, "mu"),     width=P(.05, 1.5, .05, 0.55, "width"),
            sharp=P(.5, 10, .5, 3.5, "sharp"),
            scale=P(.2, 3, .05, 0.80, "mesh um"),
            coherence=P(.05, 1, .05, 0.40, "coherence"),
        )
    ],

    # fine speckle plus striations along the long axis
    g = [
        dict(
            scope = "blob",
            w=P(0, 1, .05, .50, "weight"),      s=P(0, 2, .05, 1.50, "strength"),
            mu=P(-1, 1.5, .05, 0.10, "mu"),     width=P(.05, 1.5, .05, .60, "width"),
            sharp=P(.5, 10, .5, 7.5, "sharp"),
            scale=P(.1, 3, .05, 0.45, "grain um"),
        ),
        dict(
            scope = 'sheet',
            w=P(0, 1, .05, .60, "weight"),      s=P(0, 2, .05, 1.50, "strength"),
            mu=P(-1, 1.5, .05, 0.40, "mu"),     width=P(.05, 1.5, .05, 1.20, "width"),
            sharp=P(.5, 10, .5, 6.0, "sharp"),
            lam=P(.5, 5, .1, 1.90, "band period um"),
            coherence=P(.05, 1, .05, 0.35, "coherence"),
            len=P(1, 20, .5, 6.0, "across um"),
        ),
        dict(
            scope = 'network',
            w=P(0, 1, .05, .90, "weight"),      s=P(0, 2, .05, 1.40, "strength"),
            mu=P(-1, 1.5, .05, 0.20, "mu"),     width=P(.05, 1.5, .05, 1.10, "width"),
            sharp=P(.5, 10, .5, 7.0, "sharp"),
            scale=P(.2, 3, .05, 1.00, "mesh um"),
            coherence=P(.05, 1, .05, 0.60, "coherence"),
        )
    ],

    # nuclear: clustered chromatin plus a fine grain
    b = [
        dict(
            scope = "cluster",
            w=P(0, 1, .05, .70, "weight"),      s=P(0, 2, .05, 1.20, "strength"),
            mu=P(-1, 1.5, .05, -0.55, "mu"),    width=P(.05, 1.5, .05, .60, "width"),
            sharp=P(.5, 10, .5, 4.0, "sharp"),
            scale=P(.1, 1.5, .05, 0.30, "speckle um"),
            clust=P(.5, 8, .1, 1.40, "cluster um"),
            fill=P(.02, 1, .02, 0.35, "fill frac"),
            soft=P(.05, 1, .05, 0.30, "gate soft"),
        ),
        dict(
            scope = "blob",
            w=P(0, 1, .05, .45, "weight"),      s=P(0, 2, .05, 1.30, "strength"),
            mu=P(-1, 1.5, .05, -0.60, "mu"),    width=P(.05, 1.5, .05, .70, "width"),
            sharp=P(.5, 10, .5, 3.0, "sharp"),
            scale=P(.1, 3, .05, 0.40, "grain um"),
        ),
        dict(
            scope = 'network',
            w=P(0, 1, .05, .35, "weight"),      s=P(0, 2, .05, 1.20, "strength"),
            mu=P(-1, 1.5, .05, -.50, "mu"),     width=P(.05, 1.5, .05, .90, "width"),
            sharp=P(.5, 10, .5, 7.0, "sharp"),
            scale=P(.2, 3, .05, 0.70, "mesh um"),
            coherence=P(.05, 1, .05, 0.30, "coherence"),
        )
    ]
)
img, general = render_image(tape = tape, p_dict = marker_config, cell_mask=cell["cell"], tau = cell["tau"], phi=cell["phi"], d=cell["d"], spacing = SPACING, polar_deg = POLAR_DEG.v, azim_deg = AZIM_DEG.v,
                            um_per_vox = UM_PER_VOX)

rgb = to_rgb(cell["cell"], img)
print("RGB volume", rgb.shape, f"{rgb.nbytes/1e6:.0f} MB")

In [ ]:
ortho_rgb(rgb, title="three markers, orthogonal slices through the cell")
plt.show()

In [ ]:
fig = plot_surface_rgb_html(cell = cell, elong = ELONG.v, 
                        polar_deg = POLAR_DEG.v, azim_deg = AZIM_DEG.v, roll_deg = ROLL_DEG.v,
                        out_path = Path("../renders").resolve(), volume = rgb)

### 1.2.2) 2d Projection

In [ ]:
import psfmodels as psfm
import numpy as np
from scipy.signal import fftconvolve
from dataclasses import dataclass

In [ ]:
# Verify
FLUOROPHORES = {
    "DAPI": 0.461, "FITC": 0.519, "PE": 0.578, "APC": 0.660
}

# Optics: 
@dataclass(frozen=True)
class Optics:
    """Known Physical properties of the detector (Macsima) and the experiment."""
    um_per_px: float = 0.325 
    um_per_pz: float = 0.325 
    
    focal_um: float = 0.0
    
    # VERIFY
    # Numerical Aperture (NA)
    na: float = 0.45 #or 0.75
     
    wavelength_um: float = 0.530 # fallback
    
    # different refractive index of tissue and medium. VERIFY
    n_immersion: float = 1.0
    n_sample: float = 1.33
    
    # Thickness of the section
    section_um: float = 4.
    # Depth of the section CENTRE below the coverslip
    depth_um: float = 2.0
    
    @property
    def sample_depth_um(self):
        """Depth of the section centre below the coverslip."""
        return max(self.depth_um, self.section_um / 2)
    
    @property
    def tan_theta(self):
        return float(np.tan(np.arcsin(np.clip(self.na / self.n_immersion, 0.0, 0.999))))

optics = Optics(um_per_px=UM_PER_VOX, um_per_pz=UM_PER_VOX * Z_RATIO)
_KERNEL_CACHE = {}

def plane_heights(nz, um_per_pz):
    """Axial coordinate of each slice of a centred volume, in microns."""
    return (np.arange(nz) - (nz - 1) / 2.0) * um_per_pz

def psf_support_px(opt, thickness_um, pad=6):
    """Odd kernel width that contains the most defocused PSF for a section of `thickness_um`."""
    r = 0.5 * thickness_um * opt.tan_theta / opt.um_per_px
    return int(2 * (int(np.ceil(r)) + pad) + 1)

def quad_weights(nz):
    """Trapezoid weights for the depth integral -- half weight on the two cut faces."""
    w = np.ones(nz, float)
    w[0] = w[-1] = 0.5
    return w

def psf_kernels(opt, nxy, wavelength_um, z_um):
    
    z_um = np.asarray(z_um, float)
    # depth below the coverslip, per plane
    depth = opt.sample_depth_um + z_um
    zf = opt.sample_depth_um + opt.focal_um

    # psfmodels requires pz >= 0. Reachable in normal use: kryostat(centre_um=c) puts the
    # slab at depth in [c, c + section_um], so ANY negative centre_um trips this.
    if depth.min() < 0:
        need = float(opt.sample_depth_um - depth.min())
        raise ValueError(
            f"plane at depth {depth.min():+.3f} um sits above the coverslip; psfmodels needs "
            f"pz >= 0. Use Optics(depth_um={need:.2f}) or more (currently "
            f"{opt.sample_depth_um:.2f}), or section closer to the centre.")
    
    # Cache the kernel per fluorophore
    key = (nxy, round(float(wavelength_um), 6), round(float(zf), 6),
           opt.um_per_px, opt.na, opt.n_immersion, opt.n_sample, z_um.tobytes())
    if key in _KERNEL_CACHE:
        return _KERNEL_CACHE[key]
    
    base = dict(nx=int(nxy), dxy=opt.um_per_px, NA=opt.na, wvl=float(wavelength_um),
                    ni=opt.n_immersion, ni0=opt.n_immersion, ns=opt.n_sample)

    ks = [np.asarray(psfm.make_psf(z=[float(zf)], pz=float(d), model="scalar", **base)[0],
                            np.float32) for d in depth]
    
    ks = [k / (k.sum() + 1e-30) for k in ks]
    _KERNEL_CACHE[key] = ks
    return ks


def psf_project(slice_vol, z_um, optics, marker = None, general = None):
    nz = slice_vol.shape[0]
    if marker is None or general is None:
        wavelength_um = optics.wavelength_um
    else:
        wavelength_um = FLUOROPHORES[general["fluorophores"][marker]]
    w = quad_weights(nz)
    nxy = psf_support_px(opt = optics, thickness_um = float(np.ptp(z_um)) + optics.um_per_pz)
    ks = psf_kernels(opt = optics, nxy = nxy, wavelength_um = wavelength_um, z_um = z_um)
    out = np.zeros(slice_vol.shape[1:], np.float32)
    for i in range(nz):
        if w[i] == 0:
            continue
        out += np.float32(w[i]) * fftconvolve(slice_vol[i].astype(np.float32), ks[i], mode="same")
    return np.maximum(out, 0.0) / np.float32(w.sum())

def kryostat(vol, opt, centre_um = 0.0):
    """Cut a `section_um`-thick slab. Returns (subvolume, its plane heights in um)."""
    z = plane_heights(vol.shape[0], opt.um_per_pz)
    keep = np.abs(z - centre_um) <= opt.section_um / 2
    return vol[keep], z[keep]

def NormalizeData(stack, pct=99.5):
    return np.clip(stack / (np.percentile(stack, pct) + 1e-12), 0.0, 1.0)

subs = {marker: kryostat(v, optics) for marker, v in img.items()}   # keep BOTH vol and z

img_psf = np.stack([psf_project(v, z, optics, marker, general)
                    for marker, (v, z) in subs.items()], -1)

In [ ]:
def mask_collapse(mask, z_um, opt=None, mask_pct=0.95):
    """The honest 2D footprint of a 3D object seen through the exact optics.
    """
    cov = psf_project(slice_vol = mask.astype(np.float32), 
                      z_um = z_um, optics = opt)
    flat = np.sort(cov.ravel())[::-1]
    csum = np.cumsum(flat)
    if csum[-1] <= 0:
        return np.zeros(cov.shape, bool), cov
    k = int(np.searchsorted(csum, mask_pct * csum[-1]))
    thr = flat[min(k, flat.size - 1)]
    return cov >= thr, cov

sub, sub_z = kryostat(cell["cell"], optics)
sub_n, sub_n_z = kryostat(cell["nuc"], optics)


plt.imshow(NormalizeData(img_psf))
plt.contour(sub.any(0), colors="red" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
NormalizeData(img_psf).shape

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# Pre-normalize
img_psf_norm = (img_psf - np.min(img_psf)) / (np.max(img_psf) - np.min(img_psf))

def update_image(r_low=0.0, r_high=1.0, r_gamma=1.0,
                 g_low=0.0, g_high=1.0, g_gamma=1.0,
                 b_low=0.0, b_high=1.0, b_gamma=1.0):
    
    adjusted_img = np.zeros_like(img_psf_norm)
    params = [(r_low, r_high, r_gamma), 
              (g_low, g_high, g_gamma), 
              (b_low, b_high, b_gamma)]
    
    for i, (low, high, gamma) in enumerate(params):
        c_data = img_psf_norm[:, :, i]
        denom = max(high - low, 1e-6)
        c_norm = np.clip((c_data - low) / denom, 0.0, 1.0)
        adjusted_img[:, :, i] = c_norm ** gamma

    plt.figure(figsize=(8, 8))
    plt.imshow(adjusted_img)
    plt.contour(sub.any(0), colors="red" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
    plt.axis('off')
    plt.show()

interact(update_image, 
         r_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         r_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         r_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         # Repeat for G and B...
         g_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         g_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         g_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         b_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         b_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         b_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0))
print("done")

### 1.2.2) 2d Projection + Detector Model

In [ ]:
# VERIFY
AUTOFLUOR = {"DAPI": 0.20, "FITC": 0.60, "PE": 0.30, "APC": 0.10}


def _smooth_unit(w, sigma_px):
    """Circular Gaussian low-pass of a frozen field, rescaled to zero mean and unit variance.
    """
    ny, nx = w.shape
    fy = np.fft.fftfreq(ny)[:, None]
    fx = np.fft.rfftfreq(nx)[None, :]
    H = np.exp(-2 * np.pi ** 2 * float(sigma_px) ** 2 * (fy ** 2 + fx ** 2))
    z = np.fft.irfftn(np.fft.rfftn(w) * H, s=(ny, nx), axes=(0, 1))
    z = z - z.mean()
    return (z / (z.std() + 1e-12)).astype(np.float32)


def illumination(shape, illum_cv, vignette, tape):
    """Multiplicative flat-field: a smooth random component times a radial falloff."""
    ny, nx = shape
    rnd = 1.0 + illum_cv * _smooth_unit(tape["w_ill"], 0.25 * max(ny, nx))
    yy = (np.arange(ny) - (ny - 1) / 2.0)[:, None] / max(ny / 2.0, 1)
    xx = (np.arange(nx) - (nx - 1) / 2.0)[None, :] / max(nx / 2.0, 1)
    rr2 = np.clip((yy ** 2 + xx ** 2) / 2.0, 0.0, 1.0)     # 0 at centre, 1 at the corners
    return np.maximum(rnd * (1.0 - vignette * rr2), 0.0).astype(np.float32)


def detector(img_psf, markers, general, opt, tape, quantise=False,
             autofluor = AUTOFLUOR):
    """Turn a clean PSF projection (H, W, C) into a detector frame.
    """
    e_per_unit  = general["detector"]['E_PER_UNIT'].v
    read_e      = general["detector"]['READ_E'].v
    dark_e      = general["detector"]['DARK_E'].v
    adu_per_e   = general["detector"]['ADU_PER_E'].v
    offset_adu  = general["detector"]['OFFSET_ADU'].v
    af_scale_um = general["detector"]['AF_SCALE_UM'].v
    af_cv       = general["detector"]['AF_CV'].v
    illum_cv    = general["detector"]['ILLUM_CV'].v
    vignette    = general["detector"]['VIGNETTE'].v
    bit_depth   = general["detector"]['BIT_DEPTH']
    
    img_psf = np.asarray(img_psf, np.float32)
    ny, nx, nc = img_psf.shape
    if tape.z_shot.shape != (nc, ny, nx):
        raise ValueError(
            f"sensor tape is {tape.z_shot.shape} but the image is {(nc, ny, nx)}. "
            f"Re-run tape.drawSensor(shape=({ny}, {nx}, {nc})) -- and set IMG to match.")
    
    ill = illumination((ny, nx), illum_cv, vignette, tape)
    sig_px = max(af_scale_um / opt.um_per_px, 0.5)

    adu, diag = np.empty_like(img_psf), []
    for k, m in enumerate(markers):
        fl = general["fluorophores"][m]
        af_level = float(autofluor.get(fl, 0.0))
        # autofluorescence is emitted BY the tissue, so it is illuminated like everything else
        af = af_level * np.maximum(1.0 + af_cv * _smooth_unit(tape["w_af"][k], sig_px), 0.0)

        # marker photoelectrons
        sig_e = e_per_unit * img_psf[..., k] * ill
        # autofluorescence photoelectrons
        bg_e = e_per_unit * af * ill
        e = np.maximum(sig_e + bg_e, 0.0)
        # shot noise scales as sqrt(signal): bright pixels are noisier
        noisy = e + np.sqrt(e) * tape["z_shot"][k] + read_e * tape["z_read"][k] + dark_e
        a = np.clip(noisy * adu_per_e + offset_adu, 0.0, 2 ** bit_depth - 1)
        adu[..., k] = np.rint(a) if quantise else a

        pk = float(np.percentile(sig_e, 99.9))
        bg = float(bg_e.mean())
        diag.append((m, fl, pk, bg, pk / (np.sqrt(pk + bg + read_e ** 2) + 1e-12)))
    return adu


markers = list(subs.keys())
img_adu = detector(img_psf, markers, general, optics, tape)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(NormalizeData(img_psf), origin="lower")
ax[0].set_title("clean projection (object x PSF)")
ax[1].imshow(NormalizeData(np.maximum(img_adu - general["detector"]['OFFSET_ADU'].v, 0)), origin="lower")
ax[1].set_title("detector frame (AF + shot + read)")
ax[2].imshow(NormalizeData(img_adu[..., 0]), cmap="magma", origin="lower")
ax[2].set_title(f"channel 0 ({markers[0]}, {general['fluorophores'][markers[0]]}) in ADU")
for a in ax:
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# Pre-normalize
img_psf_norm = NormalizeData(np.maximum(img_adu - general["detector"]['OFFSET_ADU'].v, 0))

def update_image(r_low=0.0, r_high=1.0, r_gamma=1.0,
                 g_low=0.0, g_high=1.0, g_gamma=1.0,
                 b_low=0.0, b_high=1.0, b_gamma=1.0):
    
    adjusted_img = np.zeros_like(img_psf_norm)
    params = [(r_low, r_high, r_gamma), 
              (g_low, g_high, g_gamma), 
              (b_low, b_high, b_gamma)]
    
    for i, (low, high, gamma) in enumerate(params):
        c_data = img_psf_norm[:, :, i]
        denom = max(high - low, 1e-6)
        c_norm = np.clip((c_data - low) / denom, 0.0, 1.0)
        adjusted_img[:, :, i] = c_norm ** gamma

    plt.figure(figsize=(8, 8))
    plt.imshow(adjusted_img)
    plt.contour(sub.any(0), colors="red" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
    plt.contour(sub.any(0), colors="red" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
    plt.axis('off')
    plt.show()

interact(update_image, 
         r_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         r_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         r_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         # Repeat for G and B...
         g_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         g_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         g_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         b_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         b_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         b_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0))
print("done")